# Saltkraftverk: volumbalanse, osmose og effekt

## Pilotprosjekt for Matematikk 1

Når ferskvann og saltvann er skilt av en membran som slipper gjennom vann, men i liten grad salt, vil vann strømme mot saltvannssiden. I et trykkretardert osmoseanlegg holdes saltvannet under trykk. Vannet som passerer membranen øker volumstrømmen på trykksiden, og deler av denne energien kan tas ut i en turbin.

Prosjektet er inspirert av en modell for trykkretardert osmose i hule, sylindriske membranfibre. Vi begynner med volumbalanser i et nettverk, går videre til en skalar ODE for direkte osmose, og avslutter med et koblet system for volumstrøm og trykk langs én membranfiber.

### Læringsmål

Etter prosjektet skal du kunne

- sette opp lineære volumbalanser fra et nettverksdiagram,
- løse et lineært system med `numpy.linalg.solve`,
- tolke residualet i en numerisk løsning,
- løse en separabel førsteordens ODE analytisk,
- bruke Eulers metode når den uavhengige variabelen er posisjon,
- implementere Eulers metode for et koblet ODE-system,
- beregne volumstrøm og nettoeffekt fra en modell,
- undersøke hvordan trykk og fiberlengde påvirker kraftproduksjonen,
- diskutere forskjellen mellom en enkel og en mer detaljert modell.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Del A: Volumbalanse og lineær algebra

## A.1 Et enkelt nettverk

Vi bruker følgende forenklede diagram. Alle strømmer er volumstrømmer.

```text
                         permeat fra ferskvann
                                  Q_p
                                   |
                                   v
saltvann Q_s ----> [ osmoseenhet ] ----> Q_b ----> [ turbin ] ----> Q_t
                         ^                                      |
                         |                                      |
                         +-------------- Q_r <------------------+
                                                                \
                                                                 \----> Q_u ut
```

Symbolene betyr:

- $Q_s$: tilført saltvann,
- $Q_p$: vann som passerer membranen,
- $Q_b$: fortynnet brinestrøm ut av osmoseenheten,
- $Q_t$: strøm gjennom turbinen,
- $Q_r$: resirkulert strøm,
- $Q_u$: strøm som forlater anlegget.

Forutsetninger:

1. Væsken behandles som inkompressibel.
2. Anlegget er i stasjonær drift.
3. Det akkumuleres ikke væske i komponentene.

Dermed gjelder ved hvert knutepunkt

$$\sum Q_{inn}=\sum Q_{ut}.$$

## Oppgave A1: Sett opp balansene

Bruk diagrammet til å forklare ligningene

$$
\begin{aligned}
Q_b-Q_r &= Q_s+Q_p,\\
-Q_b+Q_t &=0,\\
-Q_t+Q_r+Q_u &=0.
\end{aligned}
$$

En ventil resirkulerer andelen $r$ av turbinstrømmen:

$$Q_r=rQ_t.$$

Skriv de fire ligningene som ett system

$$Ax=b$$

med

$$x=(Q_b,Q_t,Q_r,Q_u)^T.$$

In [ ]:
Q_s = 1.00
Q_p = 0.25
r = 0.60

A = np.array([
    [ 1.0,  0.0, -1.0, 0.0],
    [-1.0,  1.0,  0.0, 0.0],
    [ 0.0, -1.0,  1.0, 1.0],
    [ 0.0,   -r,  1.0, 0.0]
])

b = np.array([Q_s + Q_p, 0.0, 0.0, 0.0])

# Løs systemet.
x = ...
Q_b, Q_t, Q_r, Q_u = x

print("Q_b =", Q_b)
print("Q_t =", Q_t)
print("Q_r =", Q_r)
print("Q_u =", Q_u)
print("Residual A @ x - b =", ...)

## Oppgave A2: Tolk løsningen

1. Hvorfor er $Q_t$ større enn den eksterne tilførselen $Q_s+Q_p$?
2. Vis fra balansene at
   $$Q_u=Q_s+Q_p.$$
3. Forklar hvorfor resirkuleringen kan øke interne strømmer, men ikke skape vann.
4. Hva skjer med $Q_t$ og $Q_r$ når $r$ nærmer seg 1?

## Oppgave A3: Parameterstudie av resirkulering

Beregn strømningene for flere verdier av $r$ mellom 0 og 0.9. Plott $Q_t$, $Q_r$ og $Q_u$ som funksjoner av $r$.

Diskuter hvorfor en stor resirkuleringsgrad ikke nødvendigvis er praktisk, selv om volumbalansene gir en matematisk løsning.

In [ ]:
r_verdier = np.linspace(0.0, 0.9, 10)
resultater = []

for r_i in r_verdier:
    A_i = np.array([
        [ 1.0,   0.0, -1.0, 0.0],
        [-1.0,   1.0,  0.0, 0.0],
        [ 0.0,  -1.0,  1.0, 1.0],
        [ 0.0, -r_i,    1.0, 0.0]
    ])
    resultater.append(...)

resultater = np.array(resultater)

# Plott kolonnene for Q_t, Q_r og Q_u.

## A.2 En membranfiber delt i segmenter

Vi deler én membranfiber i tre segmenter. Vannet som passerer membranen i segment $j$, betegnes $q_j$.

```text
                    q_1             q_2             q_3
                     |               |               |
                     v               v               v
Q_0 ----> [ segment 1 ] --> Q_1 --> [ segment 2 ] --> Q_2 --> [ segment 3 ] --> Q_3
```

Volumbalansen i hvert segment er

$$
\begin{aligned}
Q_1-Q_0 &=q_1,\\
Q_2-Q_1 &=q_2,\\
Q_3-Q_2 &=q_3.
\end{aligned}
$$

## Oppgave A4: Matrise for de tre segmentene

Når $Q_0,q_1,q_2,q_3$ er kjente, skriv ligningene på formen

$$A_{seg}Q=b_{seg},$$

med

$$Q=(Q_1,Q_2,Q_3)^T.$$

Løs systemet og kontroller at

$$Q_3=Q_0+q_1+q_2+q_3.$$

In [ ]:
Q0 = 1.00
q = np.array([0.08, 0.06, 0.04])

A_seg = np.array([
    [ 1.0,  0.0, 0.0],
    [-1.0,  1.0, 0.0],
    [ 0.0, -1.0, 1.0]
])

b_seg = np.array([Q0 + q[0], q[1], q[2]])

Q = ...
print("Q1, Q2, Q3 =", Q)
print("Kontroll av Q3:", ...)

## Oppgave A5: Fra differanse til derivert

Dersom hvert segment har lengde $\Delta z$, kan balansen skrives

$$
\frac{Q_{j+1}-Q_j}{\Delta z}
=
\frac{q_j}{\Delta z}.
$$

Forklar hvorfor venstresiden ligner en derivert når segmentene gjøres korte. I del B og C går vi derfor over til en kontinuerlig volumstrøm $Q(z)$ langs fiberen.

# Del B: Direkte osmose og en skalar ODE

I artikkelmodellen beskriver $G(z)$ amplituden til hastighetsprofilen inne i en hul fiber. Middelvolumstrømmen er proporsjonal med $G$:

$$
Q(z)=\frac{\pi r_0^2}{2}G(z).
$$

Her er $z$ posisjonen langs fiberen. Eulers metode fungerer på samme måte som for en tidsavhengig ODE, men steglengden måles nå i meter.

Når den hydrostatiske trykkforskjellen neglisjeres, reduseres modellen til

$$
G'(z)=\frac{A_1}{G(z)},
\qquad G(0)=G_0>0.
$$

Konstanten $A_1>0$ samler membranegenskaper, temperatur, saltkonsentrasjon, fibergeometri og inngangshastighet.

## Oppgave B1: Analytisk løsning

Multipliser ligningen med $G(z)$ og vis at

$$
\frac{1}{2}\frac{d}{dz}G(z)^2=A_1.
$$

Vis deretter at

$$
\boxed{G(z)=\sqrt{G_0^2+2A_1z}.}
$$

Forklar hvorfor $G$ og dermed volumstrømmen øker langs fiberen.

## Oppgave B2: Euler-metoden

Implementer Eulers metode for

$$G'=A_1/G.$$

Bruk først skalerte verdier

$$G_0=1.0,\qquad A_1=0.20,$$

på intervallet $0\le z\le5$.

In [ ]:
def direkte_osmose(G):
    A1 = 0.20
    return ...


def euler_skalar(f, y0, L, h):
    N = int(round(L / h))
    z = np.linspace(0.0, N*h, N + 1)
    y = np.zeros(N + 1)
    y[0] = y0

    for n in range(N):
        y[n + 1] = ...

    return z, y


G0 = 1.0
z_B, G_B = euler_skalar(direkte_osmose, G0, L=5.0, h=0.10)

## Oppgave B3: Sammenlign med eksakt løsning

Plott Euler-løsningen og den eksakte løsningen i samme figur. Gjenta med flere steglengder og beregn maksimal feil.

In [ ]:
A1 = 0.20
G_eksakt = np.sqrt(G0**2 + 2*A1*z_B)

plt.plot(z_B, G_B, label="Euler")
plt.plot(z_B, G_eksakt, "--", label="Eksakt")
plt.xlabel("Posisjon z")
plt.ylabel("Strømningsvariabel G")
plt.legend()
plt.grid()
plt.show()

print("Maksimal feil:", ...)

## Oppgave B4: Fra $G$ til volumstrøm

Velg en fiberradius $r_0$ og beregn

$$Q(z)=\frac{\pi r_0^2}{2}G(z).$$

Beregn også den relative økningen

$$\frac{Q(z)-Q(0)}{Q(0)}.$$

Hvorfor er den relative økningen den samme for $Q$ og $G$?

In [ ]:
r0 = 50e-6  # 50 mikrometer
Q_B = ...
relativ_okning = ...

# Del C: Trykkretardert osmose

I et saltkraftverk er saltvannet satt under trykk. Det hydrostatiske trykket motvirker osmosen, mens friksjon gjør at trykket avtar langs fiberen.

Vi bruker systemet

$$
\begin{aligned}
G'(z)&=\frac{A_1}{G(z)}+A_2p(z),\\
p'(z)&=-A_3G(z),
\end{aligned}
$$

med

$$
A_1=\frac{8L_{filt}L_{refl}RTv_0c_0}{r_0},
\qquad
A_2=-\frac{4L_{filt}}{r_0},
\qquad
A_3=\frac{4\mu}{r_0^2}.
$$

Tolkning:

- $A_1/G$ beskriver osmotisk tilførsel av vann,
- $A_2p$ reduserer tilførselen når mottrykket øker,
- $-A_3G$ beskriver friksjonsbetinget trykkfall.

Siden $A_2<0$, kan mottrykket gjøre $G'$ negativt dersom trykket blir større enn den effektive osmotiske drivkraften.

## Oppgave C1: Skalert system

Vi begynner med skalerte parametre for å kunne konsentrere oss om strukturen:

$$A_1=0.20,\qquad A_2=-0.10,\qquad A_3=0.08.$$

Bruk

$$G(0)=1.0,\qquad p(0)=1.0.$$

Implementer systemet og løs det med Eulers metode.

In [ ]:
A1 = 0.20
A2 = -0.10
A3 = 0.08


def pro_system(x):
    G, p = x
    dG = ...
    dp = ...
    return np.array([dG, dp])


def euler_system(f, x0, L, h):
    N = int(round(L / h))
    z = np.linspace(0.0, N*h, N + 1)
    X = np.zeros((N + 1, len(x0)))
    X[0] = x0

    for n in range(N):
        X[n + 1] = ...

    return z, X


z_C, X_C = euler_system(
    pro_system,
    x0=np.array([1.0, 1.0]),
    L=5.0,
    h=0.01
)

G_C = X_C[:, 0]
p_C = X_C[:, 1]

In [ ]:
fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].plot(z_C, G_C)
ax[0].set_ylabel("G")
ax[0].grid()

ax[1].plot(z_C, p_C)
ax[1].set_xlabel("Posisjon z")
ax[1].set_ylabel("Trykk p")
ax[1].grid()

plt.show()

## Oppgave C2: Kvalitativ kontroll

1. Avtar trykket langs fiberen?
2. Øker eller avtar $G$?
3. Finn eventuelle punkter der $G'(z)=0$.
4. Undersøk om løsningen holder $G(z)>0$. Hvorfor er dette nødvendig både matematisk og fysisk?
5. Gjenta med mindre steglengde. Endres resultatet vesentlig?

## Oppgave C3: Volumstrøm og fortynning

Volumstrømmen er

$$Q(z)=\frac{\pi r_0^2}{2}G(z).$$

Når membranen holder saltet tilbake, er saltstrømmen langs fiberen tilnærmet konstant. Derfor kan saltkonsentrasjonen uttrykkes som

$$c(z)=c_0\frac{Q(0)}{Q(z)}=c_0\frac{G(0)}{G(z)}.$$

Beregn den relative saltkonsentrasjonen $c(z)/c_0$. Forklar hvorfor brinen fortynnes når $G$ øker.

In [ ]:
c_relativ = ...

plt.plot(z_C, c_relativ)
plt.xlabel("Posisjon z")
plt.ylabel("Relativ saltkonsentrasjon c/c0")
plt.grid()
plt.show()

## Oppgave C4: En enkel effektmodell

I den enkleste lokale modellen er produsert effekt proporsjonal med trykk ganger den ekstra volumstrømmen:

$$P_{brutto}(z)=\eta_{turb}\,[Q(z)-Q(0)]p(z).$$

En forenklet pumpekostnad kan skrives

$$P_{pumpe}=\frac{Q(0)p(0)}{\eta_{pumpe}}.
$$

For artikkelens grunnsystem brukes et mer presist uttrykk der turbin- og pumpestrømmene behandles særskilt. I første omgang bruker vi den pedagogiske modellen ovenfor.

Beregn nettoeffekten

$$P_{netto}(z)=P_{brutto}(z)-P_{pumpe}.$$

Siden de skalerte parameterne ikke har fysiske enheter, skal resultatet foreløpig bare tolkes kvalitativt.

In [ ]:
eta_turb = 0.90
eta_pumpe = 0.80

# I den skalerte modellen kan konstanten pi*r0^2/2 settes lik 1.
Q_C = G_C
Q0 = Q_C[0]
p0 = p_C[0]

P_brutto = ...
P_pumpe = ...
P_netto = ...

plt.plot(z_C, P_netto)
plt.axhline(0.0, color="black", linewidth=1)
plt.xlabel("Fiberlengde z")
plt.ylabel("Skalert nettoeffekt")
plt.grid()
plt.show()

indeks_best = ...
print("Omtrent optimal fiberlengde:", ...)
print("Maksimal skalert nettoeffekt:", ...)

## Oppgave C5: Optimalt inngangstrykk

I en svært forenklet modell med konstant osmotisk trykk $\pi_{eff}$, uten fortynning, friksjon eller virkningsgradstap, er

$$
\frac{P}{A}=L_{filt}\,p(\pi_{eff}-p).
$$

1. Deriver uttrykket med hensyn på $p$.
2. Vis at
   $$p_{opt}=\frac{\pi_{eff}}{2}.$$
3. Bruk det koblede ODE-systemet til å undersøke flere inngangstrykk $p(0)$.
4. Finn hvilket inngangstrykk som gir størst nettoeffekt i den valgte parameterstudien.
5. Diskuter hvorfor resultatet kan avvike fra halvtrykksregelen.

In [ ]:
p0_verdier = np.linspace(0.1, 2.0, 30)
beste_effekt = []
beste_lengde = []

for p0_i in p0_verdier:
    z_i, X_i = euler_system(
        pro_system,
        x0=np.array([1.0, p0_i]),
        L=5.0,
        h=0.01
    )

    G_i = X_i[:, 0]
    p_i = X_i[:, 1]

    # Beregn nettoeffekt, finn maksimum og lagre resultatet.
    # Pass også på om modellen gir G <= 0 eller andre ugyldige verdier.

# Plott maksimal nettoeffekt mot inngangstrykk.

# Del D: Fysiske parametre og modellkritikk

## D.1 Parameterverdier fra artikkelen

Artikkelen bruker blant annet eksempelverdiene

$$
L_{filt}=10^{-11}\ \mathrm{m/(s\,Pa)},
\qquad
r_0=50\ \mu\mathrm m,
\qquad
\mu=8.55\cdot10^{-4}\ \mathrm{Pa\,s},
$$

$$
\eta_{turb}=0.90,
\qquad
\eta_{pumpe}=0.80,
$$

og en effektiv osmotisk trykkforskjell på omtrent $25$ bar i flere beregninger.

En senere versjon av prosjektet kan la studentene beregne $A_1,A_2,A_3$ fra fysiske data. Før dette gjøres må alle størrelser konverteres konsekvent til SI-enheter, og det må avklares nøyaktig hvilken hastighets- og volumstrømsdefinisjon som brukes.

## Oppgave D1: Enhetskontroll

Undersøk enhetene til $A_1$, $A_2$ og $A_3$. Kontroller at begge ligningene i systemet er dimensjonalt konsistente.

## D.2 Modellkritikk

Diskuter minst fire av disse forenklingene:

- Membranen antas å holde nesten alt salt tilbake.
- Salttransport gjennom membranen neglisjeres.
- Trykktapet på ferskvannssiden neglisjeres.
- Fiberne betraktes enkeltvis, mens et virkelig anlegg har mange tettpakkede fibre.
- Temperatur og viskositet antas konstante.
- Inntaks- og utløpstap neglisjeres.
- Konsentrasjonspolarisering samles i en effektiv refleksjonskoeffisient.
- Den pedagogiske effektmodellen er enklere enn anleggsmodellen i artikkelen.

## Videreføring til Matematikk 2 og 3

I Matematikk 2 kan man undersøke konsentrasjon som funksjon av både radial og langsgående posisjon. Dette leder til konveksjons- og diffusjonsligninger. I Matematikk 3 kan man studere fluksfelt, divergens, mer fullstendige strømningmodeller og koblingen til Navier-Stokes-ligningen.

# Oppsummering

Skriv en kort rapport der du forklarer:

1. hvordan volumbalansene ga et lineært system,
2. hvordan segmentbalansen peker mot en derivert langs fiberen,
3. hvordan den skalare modellen for direkte osmose ble løst analytisk og numerisk,
4. hvordan trykk og volumstrøm kobles i systemet for trykkretardert osmose,
5. hvorfor det kan finnes både en optimal fiberlengde og et optimalt inngangstrykk,
6. hvilke forenklinger som betyr mest dersom modellen skal brukes på et virkelig anlegg.